## התאמה ליניארית: מהנוסחה לקוד

בסעיף 12.2 הערך הטוב ביותר של פרמטר אחד נמצא בסריקה על פני רשימת ערכים. לקו ישר יש שני פרמטרים, החיתוך ‎`a0`‎ והשיפוע ‎`a1`‎, ועבורו קיימת נוסחה סגורה שנותנת את שניהם ישירות, בלי סריקה: נוסחת הריבועים הפחותים (least squares). בעמוד זה הנוסחה נכתבת בקוד, בפונקציה ‎`linear_fit`‎.

המודל התאורטי לטווח הזריקה, ‎`R = (v0**2/g) * sin(2*theta)`‎, הוא קו ישר ‎`y = a0 + a1*x`‎, כאשר ‎`x = sin(2*theta)`‎ ו-‎`y = R`‎. השיפוע ‎`a1`‎ אמור לצאת קרוב ל-‎`v0**2/g`‎, והחיתוך ‎`a0`‎ קרוב ל-0.

```{admonition} 🎥 כאן נכנס סרטון השבוע
:class: seealso

**"התאמה ליניארית: מהנוסחה לקוד"** (כ-6–10 דקות, הקלטת מסך של המחברת עם קול). בסרטון בונים את פונקציית ההתאמה הליניארית מאפס, שורה אחר שורה, ומיישמים אותה על נתוני הטווח מול sin(2*theta).

<!-- TODO (למפיק/ה): להחליף תא זה בהטמעת הסרטון בפועל, למשל:
<iframe width="100%" height="400" src="VIDEO_URL_HERE" title="שבוע 12 — התאמה ליניארית: מהנוסחה לקוד" frameborder="0" allowfullscreen></iframe>
-->
```

### נוסחת הריבועים הפחותים (least squares)

לקו ‎`y = a0 + a1*x`‎ שממזער את סכום ריבועי השאריות, יש נוסחה סגורה:

$$a_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2} \qquad a_0 = \bar{y} - a_1\bar{x}$$

הנוסחה בנויה מממוצעים ומסטיות מהממוצע, כמו סטיית התקן בסעיף 9.7, אבל הפעם של שני משתנים יחד: במונה מכפלות הסטיות של ‎`x`‎ ושל ‎`y`‎, ובמכנה ריבועי הסטיות של ‎`x`‎.

In [ ]:
import numpy as np

def linear_fit(x, y):
    """Least-squares line y = a0 + a1*x. Returns a0, a1."""
    x_bar = x.mean()
    y_bar = y.mean()
    a1 = ((x - x_bar) * (y - y_bar)).sum() / ((x - x_bar)**2).sum()
    a0 = y_bar - a1 * x_bar
    return a0, a1

### יישום על נתוני הזריקה

בונים את ‎`x`‎ (‎`sin(2*theta)`‎) ו-‎`y`‎ (הטווח הממוצע) לכל זווית, מתוך ‎`lab_measurements.csv`‎.

In [ ]:
import pandas as pd

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()

angles = sorted(df_clean["angle_deg"].unique())
x = np.array([np.sin(2*np.radians(a)) for a in angles])
y = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean() for a in angles])

a0, a1 = linear_fit(x, y)
print(f"a0 = {a0:.3f}   (theory: 0)")
print(f"a1 = {a1:.3f}   (theory: v0**2/g = {20.0**2/g:.3f})")

### באג נפוץ: לשכוח למרכז (centering) את הנתונים

טעות שכיחה: לכתוב ‎`a1 = sum(x*y) / sum(x*x)`‎ -- זו נוסחת התאמה ליניארית **דרך הראשית** (‎`a0 = 0`‎ בכפייה), לא ההתאמה הכללית. היא לא זורקת שגיאה, ונותנת מספר סביר -- אבל שגוי, כי היא מתעלמת מהחיתוך. שתי הנוסחאות עונות על שאלות שונות: קו שעובר בכפייה דרך הראשית, מול קו עם חיתוך חופשי. במודל של הטווח החיתוך חופשי, ולכן הנוסחה המתאימה היא זו עם המרכוז.

In [ ]:
a1_through_origin = np.sum(x*y) / np.sum(x*x)   # assumes (wrongly) a0 = 0
print(f"slope through the origin: {a1_through_origin:.3f}")
print(f"slope with centering:     {a1:.3f}")

### נסו בעצמכם

הפעילו את ‎`linear_fit`‎ על ‎`run_id`‎ (ציר x) מול ‎`v0_measured`‎ (ציר y) בתוך זווית 45 בלבד. השיפוע ‎`a1_45`‎ אמור לצאת קרוב מאוד ל-0 (אין מגמת שינוי אמיתית לאורך הריצות -- v0 היה אמור להיות קבוע).

In [ ]:
# sub45 = df_clean[df_clean["angle_deg"] == 45]
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
sub45 = df_clean[df_clean["angle_deg"] == 45]
a0_45, a1_45 = linear_fit(sub45["run_id"], sub45["v0_measured"])
print(f"a1_45 = {a1_45:.5f}")
```
השיפוע יוצא כ-0.00003, קרוב מאוד ל-0, כצפוי: ‎`v0`‎ אינו אמור להשתנות עם ‎`run_id`‎.
`````

### בדקו את עצמכם

השאלה מתייחסת לשתי השורות הבאות, שתיהן לחישוב השיפוע:

```python
a1 = ((x - x_bar) * (y - y_bar)).sum() / ((x - x_bar)**2).sum()   # line 1
a1 = (x*y).sum() / (x*x).sum()                                    # line 2
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "מה ההבדל בין שתי השורות?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "אין הבדל, שתי השורות נותנות תמיד אותו שיפוע",
                "correct": false,
                "feedback": "לא - הן נותנות אותו שיפוע רק כאשר הממוצע של x הוא 0."
            },
            {
                "answer": "השורה השנייה נכונה תמיד, והראשונה שגויה",
                "correct": false,
                "feedback": "לא - שתיהן נכונות, לשתי שאלות שונות."
            },
            {
                "answer": "שתי השורות מחשבות את החיתוך a0, ולא את השיפוע a1",
                "correct": false,
                "feedback": "לא - שתיהן מחשבות את השיפוע a1; ההבדל ביניהן הוא בהנחה על החיתוך a0."
            },
            {
                "answer": "הראשונה מתאימה קו כללי (a0 חופשי), והשנייה מכריחה את הקו לעבור דרך הראשית (a0 = 0)",
                "correct": true,
                "feedback": "נכון."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)